In [2]:
# NOTE: This file is the source for four Jupyter notebook cells.
# Copy each section (between the ╔═╗ banners) into its own cell.
# =============================================================================
#
#  INTERCONNECT — 14-Ring Add-Drop Cascade  ·  neff / ng Parametric Sweep
#  ─────────────────────────────────────────────────────────────────────────
#  Circuit topology
#  ─────────────────
#  ONA_1 output   ──► RING_1  port 1  (bus input)
#  RING_1  through ──► ONA    input 1
#  RING_1  drop    ──► RING_2  port 1
#  RING_2  through ──► terminated (unused through arm)
#  RING_2  drop    ──► OPM_1
#  RING_3  drop    ──► OPM_2
#   … (drop chain continues) …
#  RING_14 through ──► ONA    input 2
#  RING_14 drop    ──► OPM_13
#
#  Sweep axis
#  ──────────
#  Paired (neff_TE, ng_TE) values for RING_1 only.
#  All other rings are held at their fixed parameter values.
#
# =============================================================================


# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 1 — Imports · lumapi setup · Logging · I/O paths                     ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

import sys
import os
import platform
import time
import logging
from pathlib import Path
from datetime import datetime
from dataclasses import dataclass, asdict
from typing import List, Optional, Tuple, Dict, Any

import numpy as np
import h5py
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable

# ─────────────────────────────────────────────────────────────────────────────
# Lumerical installation paths  (identical logic to the FDE notebook Cell 1)
# ─────────────────────────────────────────────────────────────────────────────
LUMERICAL_VERSION = "v202"          # ← match your installation folder exactly

if platform.system() == "Windows":
    LUMERICAL_ROOT = rf"C:\Program Files\Lumerical\{LUMERICAL_VERSION}"
    LUMERICAL_API  = rf"{LUMERICAL_ROOT}\api\python"
    LUMERICAL_BIN  = rf"{LUMERICAL_ROOT}\bin"
else:
    LUMERICAL_ROOT = f"/opt/lumerical/{LUMERICAL_VERSION}"
    LUMERICAL_API  = f"{LUMERICAL_ROOT}/api/python"
    LUMERICAL_BIN  = f"{LUMERICAL_ROOT}/bin"

# ── Clear any previously cached failed import ────────────────────────────────
if "lumapi" in sys.modules:
    del sys.modules["lumapi"]

# ── Add API folder to sys.path ────────────────────────────────────────────────
if LUMERICAL_API not in sys.path:
    sys.path.insert(0, LUMERICAL_API)

# ── Register bin as DLL search path (Python 3.8+, Windows) ───────────────────
if platform.system() == "Windows":
    if hasattr(os, "add_dll_directory"):
        os.add_dll_directory(str(LUMERICAL_BIN))
    else:
        os.environ["PATH"] = str(LUMERICAL_BIN) + ";" + os.environ.get("PATH", "")

# ── Verify paths ──────────────────────────────────────────────────────────────
assert Path(LUMERICAL_API).exists(), (
    f"Lumerical API path not found:\n  {LUMERICAL_API}\n"
    f"Check LUMERICAL_VERSION = '{LUMERICAL_VERSION}'"
)
assert Path(LUMERICAL_BIN).exists(), f"Lumerical bin path not found:\n  {LUMERICAL_BIN}"

import lumapi  # noqa — must come after all path setup above
print(f"lumapi imported successfully from:\n  {lumapi.__file__}")

# ─────────────────────────────────────────────────────────────────────────────
# Logging
# ─────────────────────────────────────────────────────────────────────────────
logging.basicConfig(
    level   = logging.INFO,
    format  = "%(asctime)s │ %(levelname)s │ %(message)s",
    datefmt = "%H:%M:%S",
)
log = logging.getLogger("ICNT_Cascade")

# ─────────────────────────────────────────────────────────────────────────────
# I/O paths
# ─────────────────────────────────────────────────────────────────────────────
VERSION_NAME = "ICNT_14Ring_Cascade_neff_sweep_V1"
PROJECT_DIR  = Path.cwd()
DATA_DIR     = PROJECT_DIR / "data_ICNT_cascade_ring_sweep"
DATA_DIR.mkdir(parents=True, exist_ok=True)
HDF5_PATH    = DATA_DIR / f"{VERSION_NAME}.h5"
FIGURES_DIR  = DATA_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"\n  Data directory : {DATA_DIR}")
print(f"  HDF5 output    : {HDF5_PATH}")
print(f"  Figures dir    : {FIGURES_DIR}")

# ═════════════════════════════════════════════════════════════════════════════
#  END CELL 1
# ═════════════════════════════════════════════════════════════════════════════


# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 2 — All simulation parameters defined as arrays                      ║
# ║                                                                            ║
# ║  ► Edit the values in this cell only.                                      ║
# ║  ► One entry per ring (index 0 = Ring 1, index 13 = Ring 14).              ║
# ║  ► All length / wavelength quantities in SI metres unless noted.           ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

N_RINGS = 14   # total number of add-drop rings in the cascade

# ─────────────────────────────────────────────────────────────────────────────
#  PER-RING PARAMETER ARRAYS
#  Length = N_RINGS  (one value per ring, index 0 → Ring 1, index 13 → Ring 14)
# ─────────────────────────────────────────────────────────────────────────────

# ── Geometry ──────────────────────────────────────────────────────────────────
# Ring radius [m]
RING_RADIUS_M = np.array([
    19.0021e-6,   # Ring  1
    19.1818e-6,   # Ring  2
    19.1934e-6,   # Ring  3
    19.2051e-6,   # Ring  4
    19.2168e-6,   # Ring  5
    19.2284e-6,   # Ring  6
    19.2401e-6,   # Ring  7
    19.2517e-6,   # Ring  8
    19.4158e-6,   # Ring  9
    19.4275e-6,   # Ring 10
    19.4393e-6,   # Ring 11
    19.4511e-6,   # Ring 12
    19.4628e-6,   # Ring 13
    19.4746e-6,   # Ring 14
])

# ── Resonance / expansion wavelength [m] ──────────────────────────────────────
RING_LAMBDA_RES_M = np.array([
    1.550e-6,   # Ring  1
    1.550e-6,   # Ring  2
    1.5507692e-6,   # Ring  3
    1.5515385e-6,   # Ring  4
    1.5523077e-6,   # Ring  5
    1.5530769e-6,   # Ring  6
    1.5538462e-6,   # Ring  7
    1.5546154e-6,   # Ring  8
    1.5553846e-6,   # Ring  9
    1.5561538e-6,   # Ring 10
    1.5569231e-6,   # Ring 11
    1.5576923e-6,   # Ring 12
    1.5584615e-6,   # Ring 13
    1.5582308e-6,   # Ring 14
])

# ── TE effective index ─────────────────────────────────────────────────────────
RING_NEFF_TE = np.array([
    1.609803,   # Ring  1   ← swept in Cell 3; this is the fixed default
    1.633303,   # Ring  2
    1.633121,   # Ring  3
    1.632939,   # Ring  4
    1.632758,   # Ring  5
    1.632576,   # Ring  6
    1.632394,   # Ring  7
    1.632213,   # Ring  8
    1.631974,   # Ring  9
    1.631792,   # Ring 10
    1.631611,   # Ring 11
    1.631430,   # Ring 12
    1.631248,   # Ring 13
    1.631067,   # Ring 14
])

# ── TE group index ─────────────────────────────────────────────────────────────
RING_NG_TE = np.array([
    2.020543,   # Ring  1   ← swept in Cell 3; this is the fixed default
    1.991101,   # Ring  2
    1.990956,   # Ring  3
    1.990808,   # Ring  4
    1.990659,   # Ring  5
    1.990509,   # Ring  6
    1.990356,   # Ring  7
    1.990203,   # Ring  8
    1.990047,   # Ring  9
    1.989891,   # Ring 10
    1.989733,   # Ring 11
    1.989691,   # Ring 12
    1.989528,   # Ring 13
    1.989364,   # Ring 14
])

# ── TE GVD dispersion [ps²/km]  (converted to s²/m internally: × 1e-15) ───────
RING_D_TE_PS2_PER_KM = np.array([
    0.0,   # Ring  1
    0.0,   # Ring  2
    0.0,   # Ring  3
    0.0,   # Ring  4
    0.0,   # Ring  5
    0.0,   # Ring  6
    0.0,   # Ring  7
    0.0,   # Ring  8
    0.0,   # Ring  9
    0.0,   # Ring 10
    0.0,   # Ring 11
    0.0,   # Ring 12
    0.0,   # Ring 13
    0.0,   # Ring 14
])

# ── TM effective index ─────────────────────────────────────────────────────────
RING_NEFF_TM = np.array([
    1.7000,   # Ring  1
    1.7000,   # Ring  2
    1.7000,   # Ring  3
    1.7000,   # Ring  4
    1.7000,   # Ring  5
    1.7000,   # Ring  6
    1.7000,   # Ring  7
    1.7000,   # Ring  8
    1.7000,   # Ring  9
    1.7000,   # Ring 10
    1.7000,   # Ring 11
    1.7000,   # Ring 12
    1.7000,   # Ring 13
    1.7000,   # Ring 14
])

# ── TM group index ─────────────────────────────────────────────────────────────
RING_NG_TM = np.array([
    2.1000,   # Ring  1
    2.1000,   # Ring  2
    2.1000,   # Ring  3
    2.1000,   # Ring  4
    2.1000,   # Ring  5
    2.1000,   # Ring  6
    2.1000,   # Ring  7
    2.1000,   # Ring  8
    2.1000,   # Ring  9
    2.1000,   # Ring 10
    2.1000,   # Ring 11
    2.1000,   # Ring 12
    2.1000,   # Ring 13
    2.1000,   # Ring 14
])

# ── TM GVD dispersion [ps²/km] ────────────────────────────────────────────────
RING_D_TM_PS2_PER_KM = np.array([
    0.0,   # Ring  1
    0.0,   # Ring  2
    0.0,   # Ring  3
    0.0,   # Ring  4
    0.0,   # Ring  5
    0.0,   # Ring  6
    0.0,   # Ring  7
    0.0,   # Ring  8
    0.0,   # Ring  9
    0.0,   # Ring 10
    0.0,   # Ring 11
    0.0,   # Ring 12
    0.0,   # Ring 13
    0.0,   # Ring 14
])

# ── Input coupler power coupling coefficient  |κ|² ────────────────────────────
RING_KAPPA_INPUT_SQ = np.array([
    0.145778,   # Ring  1
    0.145072,   # Ring  2
    0.145011,   # Ring  3
    0.144949,   # Ring  4
    0.144888,   # Ring  5
    0.144827,   # Ring  6
    0.144765,   # Ring  7
    0.144704,   # Ring  8
    0.145696,   # Ring  9
    0.145634,   # Ring 10
    0.145572,   # Ring 11
    0.145518,   # Ring 12
    0.145456,   # Ring 13
    0.145394,   # Ring 14
])

# ── Drop coupler power coupling coefficient  |κ|² ─────────────────────────────
RING_KAPPA_DROP_SQ = np.array([
    0.143402,   # Ring  1
    0.142672,   # Ring  2
    0.142609,   # Ring  3
    0.142546,   # Ring  4
    0.142484,   # Ring  5
    0.142420,   # Ring  6
    0.142357,   # Ring  7
    0.142294,   # Ring  8
    0.143269,   # Ring  9
    0.143205,   # Ring 10
    0.143142,   # Ring 11
    0.143086,   # Ring 12
    0.143022,   # Ring 13
    0.142958,   # Ring 14
])

# ── Propagation loss [dB/m]  (100 dB/m ≈ 1 dB/cm, typical SiN) ──────────────
RING_LOSS_DB_PER_M = np.array([
    101.0,   # Ring  1
    101.0,   # Ring  2
    101.0,   # Ring  3
    101.0,   # Ring  4
    101.0,   # Ring  5
    101.0,   # Ring  6
    101.0,   # Ring  7
    101.0,   # Ring  8
    101.0,   # Ring  9
    101.0,   # Ring 10
    101.0,   # Ring 11
    101.0,   # Ring 12
    101.0,   # Ring 13
    101.0,   # Ring 14
])

# ── Active polarization used in the INTERCONNECT ring model ───────────────────
# One string per ring: "TE" or "TM"
RING_POLARIZATION = [
    "TE",   # Ring  1
    "TE",   # Ring  2
    "TE",   # Ring  3
    "TE",   # Ring  4
    "TE",   # Ring  5
    "TE",   # Ring  6
    "TE",   # Ring  7
    "TE",   # Ring  8
    "TE",   # Ring  9
    "TE",   # Ring 10
    "TE",   # Ring 11
    "TE",   # Ring 12
    "TE",   # Ring 13
    "TE",   # Ring 14
]

# ─────────────────────────────────────────────────────────────────────────────
#  ONA PARAMETERS
# ─────────────────────────────────────────────────────────────────────────────
ONA_LAMBDA_START_M  = 1.540e-6    # sweep start wavelength  [m]
ONA_LAMBDA_STOP_M   = 1.560e-6    # sweep stop  wavelength  [m]
ONA_N_POINTS        = 1000        # number of wavelength/frequency samples
ONA_POWER_DBM       = 0.0         # source input power  [dBm]
ONA_N_INPUT_PORTS   = 2           # number of ONA input ports

# ─────────────────────────────────────────────────────────────────────────────
#  SWEEP PARAMETERS  — paired (neff_TE, ng_TE) for RING_1 only
#
#  Both arrays must have the same length = SWEEP_N_POINTS.
#  They are treated as paired: sweep point k uses
#      neff = SWEEP_NEFF[k]  and  ng = SWEEP_NG[k]
#
#  Two helper patterns are shown:
#    A) np.linspace  → uniform ramp
#    B) explicit list → irregular / non-uniform steps
# ─────────────────────────────────────────────────────────────────────────────
SWEEP_N_POINTS = 21

# Pattern A — uniform ramp (most common):
SWEEP_NEFF = np.linspace(1.90, 2.10, SWEEP_N_POINTS)
SWEEP_NG   = np.linspace(2.20, 2.45, SWEEP_N_POINTS)

# Pattern B — uncomment and edit for irregular points:
# SWEEP_NEFF = np.array([1.90, 1.92, 1.95, 1.98, 2.00, 2.02, 2.05, 2.08, 2.10])
# SWEEP_NG   = np.array([2.20, 2.22, 2.25, 2.28, 2.30, 2.32, 2.35, 2.39, 2.45])
# SWEEP_N_POINTS = len(SWEEP_NEFF)   # recompute after manual override

# ─────────────────────────────────────────────────────────────────────────────
#  VALIDATION — catch obvious mistakes before running anything
# ─────────────────────────────────────────────────────────────────────────────
assert len(RING_RADIUS_M)          == N_RINGS, "RING_RADIUS_M length mismatch"
assert len(RING_LAMBDA_RES_M)      == N_RINGS, "RING_LAMBDA_RES_M length mismatch"
assert len(RING_NEFF_TE)           == N_RINGS, "RING_NEFF_TE length mismatch"
assert len(RING_NG_TE)             == N_RINGS, "RING_NG_TE length mismatch"
assert len(RING_D_TE_PS2_PER_KM)   == N_RINGS, "RING_D_TE_PS2_PER_KM length mismatch"
assert len(RING_NEFF_TM)           == N_RINGS, "RING_NEFF_TM length mismatch"
assert len(RING_NG_TM)             == N_RINGS, "RING_NG_TM length mismatch"
assert len(RING_D_TM_PS2_PER_KM)   == N_RINGS, "RING_D_TM_PS2_PER_KM length mismatch"
assert len(RING_KAPPA_INPUT_SQ)    == N_RINGS, "RING_KAPPA_INPUT_SQ length mismatch"
assert len(RING_KAPPA_DROP_SQ)     == N_RINGS, "RING_KAPPA_DROP_SQ length mismatch"
assert len(RING_LOSS_DB_PER_M)     == N_RINGS, "RING_LOSS_DB_PER_M length mismatch"
assert len(RING_POLARIZATION)      == N_RINGS, "RING_POLARIZATION length mismatch"
assert len(SWEEP_NEFF) == SWEEP_N_POINTS, "SWEEP_NEFF length ≠ SWEEP_N_POINTS"
assert len(SWEEP_NG)   == SWEEP_N_POINTS, "SWEEP_NG   length ≠ SWEEP_N_POINTS"
assert len(SWEEP_NEFF) == len(SWEEP_NG),  "SWEEP_NEFF and SWEEP_NG must be same length"

# ─────────────────────────────────────────────────────────────────────────────
#  Summary printout
# ─────────────────────────────────────────────────────────────────────────────
print("=" * 70)
print("  INTERCONNECT 14-Ring Cascade — Parameter Summary")
print("=" * 70)
print(f"  {'Ring':>5}  {'R [µm]':>8}  {'λ_res [nm]':>12}  "
      f"{'neff_TE':>8}  {'ng_TE':>7}  {'κ²_in':>7}  {'κ²_dr':>7}  {'Loss [dB/m]':>11}")
print("  " + "─" * 65)
for i in range(N_RINGS):
    sweep_tag = " ◄ swept" if i == 0 else ""
    print(f"  {i+1:>5}  {RING_RADIUS_M[i]*1e6:>8.3f}  "
          f"{RING_LAMBDA_RES_M[i]*1e9:>12.3f}  "
          f"{RING_NEFF_TE[i]:>8.5f}  {RING_NG_TE[i]:>7.5f}  "
          f"{RING_KAPPA_INPUT_SQ[i]:>7.4f}  {RING_KAPPA_DROP_SQ[i]:>7.4f}  "
          f"{RING_LOSS_DB_PER_M[i]:>11.1f}"
          f"{sweep_tag}")
print()
print(f"  ONA     : λ {ONA_LAMBDA_START_M*1e9:.2f}–{ONA_LAMBDA_STOP_M*1e9:.2f} nm  │  "
      f"{ONA_N_POINTS} pts  │  {ONA_POWER_DBM} dBm")
print(f"  Sweep   : neff {SWEEP_NEFF[0]:.4f}→{SWEEP_NEFF[-1]:.4f}  │  "
      f"ng {SWEEP_NG[0]:.4f}→{SWEEP_NG[-1]:.4f}  │  {SWEEP_N_POINTS} pts")
print(f"  Total runs : {SWEEP_N_POINTS}")
print("=" * 70)

# ═════════════════════════════════════════════════════════════════════════════
#  END CELL 2
# ═════════════════════════════════════════════════════════════════════════════


# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 3 — INTERCONNECT circuit builder + sweep engine + HDF5 storage       ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

# ─────────────────────────────────────────────────────────────────────────────
# Element name helpers  (centralised — downstream cells use these too)
# ─────────────────────────────────────────────────────────────────────────────
ONA_NAME = "ONA_1"

def ring_name(ring_id: int) -> str:
    """Return INTERCONNECT element name for ring ring_id (1-indexed)."""
    return f"RING_{ring_id}"

def opm_name(opm_id: int) -> str:
    """Return INTERCONNECT element name for OPM opm_id (1-indexed, 1..13)."""
    return f"OPM_{opm_id}"


# ─────────────────────────────────────────────────────────────────────────────
# Helper: _apply_ring_params
#   Writes all parameters from the global arrays to a named ring element
#   in the open INTERCONNECT session.
#   Called once per ring during _build_circuit, and called again per step
#   during the sweep to update Ring 1's neff / ng.
# ─────────────────────────────────────────────────────────────────────────────
def _apply_ring_params(
    ic,
    ring_idx     : int,           # 0-indexed (ring_idx=0 → Ring 1)
    neff_override: Optional[float] = None,   # replaces RING_NEFF_TE[ring_idx]
    ng_override  : Optional[float] = None,   # replaces RING_NG_TE[ring_idx]
):
    """
    Set all INTERCONNECT parameters for ring at index ring_idx.
    Reads from the global per-ring arrays defined in Cell 2.
    neff_override / ng_override are used during the sweep for Ring 1.

    INTERCONNECT ring model property names:
      "radius"                  [m]
      "center wavelength"       [m]
      "effective index 1"       dimensionless
      "group index 1"           dimensionless
      "loss 1"                  [dB/m]
      "dispersion 1"            [s²/m]   (1 ps²/km = 1e-15 s²/m)
      "coupling coefficient 1"  power coupling |κ|²  (input bus)
      "coupling coefficient 2"  power coupling |κ|²  (drop bus)
    """
    name = ring_name(ring_idx + 1)      # INTERCONNECT element name

    neff = neff_override if neff_override is not None else RING_NEFF_TE[ring_idx]
    ng   = ng_override   if ng_override   is not None else RING_NG_TE[ring_idx]

    pol = RING_POLARIZATION[ring_idx].upper()
    d_val = (RING_D_TE_PS2_PER_KM[ring_idx] if pol == "TE"
             else RING_D_TM_PS2_PER_KM[ring_idx])

    ic.setnamed(name, "radius",                 float(RING_RADIUS_M[ring_idx]))
    ic.setnamed(name, "center wavelength",      float(RING_LAMBDA_RES_M[ring_idx]))
    ic.setnamed(name, "effective index 1",      float(neff))
    ic.setnamed(name, "group index 1",          float(ng))
    ic.setnamed(name, "loss 1",                 float(RING_LOSS_DB_PER_M[ring_idx]))
    ic.setnamed(name, "dispersion 1",           float(d_val * 1e-15))
    ic.setnamed(name, "coupling coefficient 1", float(RING_KAPPA_INPUT_SQ[ring_idx]))
    ic.setnamed(name, "coupling coefficient 2", float(RING_KAPPA_DROP_SQ[ring_idx]))


# ─────────────────────────────────────────────────────────────────────────────
# Helper: _build_circuit
#   Clears the INTERCONNECT canvas and rebuilds the full 14-ring cascade.
#   Circuit topology matches the header comment.
#   Layout positions are computed automatically (display only).
# ─────────────────────────────────────────────────────────────────────────────
def _build_circuit(ic):
    """
    Build the full 14-ring add-drop cascade from scratch.

    Topology (corrected per user spec):
      ONA output       → RING_1 port 1
      RING_1 through   → ONA   input 1
      RING_n drop      → RING_{n+1} port 1   (drop-feeds-next chain)
      RING_14 through  → ONA   input 2
      RING_n drop      → OPM_{n-1}  for n = 2 … 14
    """
    ic.switchtolayout()
    ic.selectall()
    ic.delete()

    dx = 130e-6    # horizontal spacing between ring centres [m] (layout only)

    # ── ONA ───────────────────────────────────────────────────────────────────
    ic.addona()
    ic.set("name",        ONA_NAME)
    ic.set("x position",  0.0)
    ic.set("y position",  0.0)

    # Convert power from dBm → W for INTERCONNECT
    pwr_W = 10 ** (ONA_POWER_DBM / 10) * 1e-3
    ic.setnamed(ONA_NAME, "start frequency",       3e8 / ONA_LAMBDA_STOP_M)
    ic.setnamed(ONA_NAME, "stop frequency",        3e8 / ONA_LAMBDA_START_M)
    ic.setnamed(ONA_NAME, "number of points",      ONA_N_POINTS)
    ic.setnamed(ONA_NAME, "input power",           pwr_W)
    ic.setnamed(ONA_NAME, "number of input ports", ONA_N_INPUT_PORTS)

    # ── Rings ─────────────────────────────────────────────────────────────────
    for i in range(N_RINGS):
        rn = ring_name(i + 1)
        ic.addelement("Double Bus Ring Resonator")
        ic.set("name",        rn)
        ic.set("x position",  float((i + 1) * dx))
        ic.set("y position",  0.0)
        _apply_ring_params(ic, ring_idx=i)

    # ── OPMs  (OPM_1 … OPM_13, one per ring 2–14 drop) ───────────────────────
    n_opm = N_RINGS - 1
    for k in range(1, n_opm + 1):
        on = opm_name(k)
        ic.addpowermeter()
        ic.set("name",        on)
        ic.set("x position",  float((k + 1) * dx))
        ic.set("y position",  -140e-6)

    # ── Connections ───────────────────────────────────────────────────────────
    # ONA output → RING_1 port 1
    ic.connect(ONA_NAME, "output", ring_name(1), "port 1")

    # RING_1 through → ONA input 1
    ic.connect(ring_name(1), "port 2", ONA_NAME, "input 1")

    # Drop chain: RING_n drop → RING_{n+1} port 1
    for i in range(1, N_RINGS):     # i = 1 … 13
        ic.connect(ring_name(i), "port 4", ring_name(i + 1), "port 1")

    # RING_14 through → ONA input 2
    ic.connect(ring_name(N_RINGS), "port 2", ONA_NAME, "input 2")

    # RING_n drop → OPM_{n-1}  for n = 2 … 14
    for i in range(2, N_RINGS + 1):
        opm_id = i - 1
        ic.connect(ring_name(i), "port 4", opm_name(opm_id), "input")

    log.info(f"Circuit built: {N_RINGS} rings, {n_opm} OPMs, 1 ONA.")


# ─────────────────────────────────────────────────────────────────────────────
# Helper: _extract_results
#   After a completed INTERCONNECT run, reads:
#     - ONA port 1 and port 2 transmission spectra   [dB]
#     - Per-OPM spectral power arrays                [W]
#     - Per-OPM mean power scalar                    [dBm]
# ─────────────────────────────────────────────────────────────────────────────
def _extract_results(ic) -> Tuple[
    np.ndarray,   # wavelengths_m   (n_wl,)
    np.ndarray,   # T_port1_dB      (n_wl,)
    np.ndarray,   # T_port2_dB      (n_wl,)
    np.ndarray,   # opm_power_dBm   (n_opm,)
    np.ndarray,   # opm_spectrum_W  (n_opm, n_wl)
]:
    n_opm = N_RINGS - 1

    # ── Wavelength axis from ONA port 1 ───────────────────────────────────────
    raw_1  = ic.getresult(ONA_NAME, "input 1/mode 1/transmission")
    f_arr  = np.asarray(raw_1["f"]).flatten()
    # Sort ascending in wavelength (descending in frequency)
    sort_i = np.argsort(f_arr)[::-1]
    wl_m   = 3e8 / f_arr[sort_i]

    def _T_dB(port_label: str) -> np.ndarray:
        raw  = ic.getresult(ONA_NAME, f"{port_label}/mode 1/transmission")
        T    = np.asarray(raw["T"]).flatten()[sort_i]
        T    = np.where(T > 0, T, 1e-30)
        return 10.0 * np.log10(np.abs(T))

    T_port1_dB = _T_dB("input 1")
    T_port2_dB = _T_dB("input 2")

    # ── OPM spectra ───────────────────────────────────────────────────────────
    n_wl           = len(wl_m)
    opm_power_dBm  = np.full(n_opm, np.nan)
    opm_spectrum_W = np.full((n_opm, n_wl), np.nan)

    for k in range(1, n_opm + 1):
        try:
            raw_p = ic.getresult(opm_name(k), "power")
            p_arr = np.asarray(raw_p).flatten()
            if p_arr.size == n_wl:
                opm_spectrum_W[k - 1, :] = p_arr[sort_i] if p_arr.size == f_arr.size else p_arr
            elif p_arr.size > 0:
                opm_spectrum_W[k - 1, :] = p_arr[0]
            mean_p = np.nanmean(opm_spectrum_W[k - 1, :])
            opm_power_dBm[k - 1] = 10.0 * np.log10(mean_p * 1e3 + 1e-40)
        except Exception as exc:
            log.warning(f"  OPM {k} extraction failed: {exc}")

    return wl_m, T_port1_dB, T_port2_dB, opm_power_dBm, opm_spectrum_W


# ─────────────────────────────────────────────────────────────────────────────
# Helper: _init_hdf5
#   Creates the HDF5 file with pre-allocated, NaN-initialised datasets.
#   Schema mirrors the FDE HDF5 file: metadata / results / flags.
# ─────────────────────────────────────────────────────────────────────────────
def _init_hdf5(wl_ref_m: np.ndarray):
    """
    Create a new HDF5 file.  Called on the first successful simulation run,
    once the real wavelength axis (wl_ref_m) is known.
    """
    n_pts  = SWEEP_N_POINTS
    n_opm  = N_RINGS - 1
    n_wl   = len(wl_ref_m)

    with h5py.File(HDF5_PATH, "w") as f:
        # ── metadata ──────────────────────────────────────────────────────────
        md = f.create_group("metadata")
        md.create_dataset("neff_sweep",    data=SWEEP_NEFF)
        md.create_dataset("ng_sweep",      data=SWEEP_NG)
        md.create_dataset("wavelengths_m", data=wl_ref_m)
        md.attrs["version_name"]      = VERSION_NAME
        md.attrs["n_rings"]           = N_RINGS
        md.attrs["n_opm"]             = n_opm
        md.attrs["sweep_n_points"]    = SWEEP_N_POINTS
        md.attrs["ona_lambda_start_m"]= ONA_LAMBDA_START_M
        md.attrs["ona_lambda_stop_m"] = ONA_LAMBDA_STOP_M
        md.attrs["ona_n_points"]      = ONA_N_POINTS
        md.attrs["ona_power_dBm"]     = ONA_POWER_DBM
        md.attrs["timestamp_start"]   = datetime.now().isoformat()

        # Per-ring parameters stored as attributes
        for i in range(N_RINGS):
            pfx = f"ring{i+1}_"
            md.attrs[pfx + "radius_m"]          = RING_RADIUS_M[i]
            md.attrs[pfx + "lambda_res_m"]      = RING_LAMBDA_RES_M[i]
            md.attrs[pfx + "neff_TE"]           = RING_NEFF_TE[i]
            md.attrs[pfx + "ng_TE"]             = RING_NG_TE[i]
            md.attrs[pfx + "D_TE_ps2_per_km"]   = RING_D_TE_PS2_PER_KM[i]
            md.attrs[pfx + "neff_TM"]           = RING_NEFF_TM[i]
            md.attrs[pfx + "ng_TM"]             = RING_NG_TM[i]
            md.attrs[pfx + "D_TM_ps2_per_km"]   = RING_D_TM_PS2_PER_KM[i]
            md.attrs[pfx + "kappa_input_sq"]    = RING_KAPPA_INPUT_SQ[i]
            md.attrs[pfx + "kappa_drop_sq"]     = RING_KAPPA_DROP_SQ[i]
            md.attrs[pfx + "loss_dB_per_m"]     = RING_LOSS_DB_PER_M[i]
            md.attrs[pfx + "polarization"]      = RING_POLARIZATION[i]

        # ── results (NaN-initialised, chunked per sweep point) ─────────────────
        rg = f.create_group("results")
        rg.create_dataset("T_port1_dB",
                          data=np.full((n_pts, n_wl),        np.nan),
                          chunks=(1, n_wl))
        rg.create_dataset("T_port2_dB",
                          data=np.full((n_pts, n_wl),        np.nan),
                          chunks=(1, n_wl))
        rg.create_dataset("opm_power_dBm",
                          data=np.full((n_pts, n_opm),       np.nan),
                          chunks=(1, n_opm))
        rg.create_dataset("opm_spectrum_W",
                          data=np.full((n_pts, n_opm, n_wl), np.nan),
                          chunks=(1, n_opm, n_wl))

        # ── progress flags ─────────────────────────────────────────────────────
        f.create_group("flags").create_dataset(
            "computed",
            data=np.zeros(n_pts, dtype=bool),
            chunks=(1,),
        )

    log.info(f"HDF5 initialised → {HDF5_PATH}")


# ─────────────────────────────────────────────────────────────────────────────
# MAIN SWEEP FUNCTION: run_interconnect_sweep
# ─────────────────────────────────────────────────────────────────────────────
def run_interconnect_sweep(hide_gui: bool = False) -> Dict[str, Any]:
    """
    Run the parametric (neff_TE, ng_TE) sweep for RING_1 in the 14-ring
    INTERCONNECT cascade.

    All parameters are read from the global arrays defined in Cell 2.
    Ring 2–14 are held fixed; only Ring 1 neff / ng change per sweep point.

    Each step:
      1. Switches to layout, updates Ring 1 neff / ng via setnamed.
      2. Runs the simulation.
      3. Extracts ONA spectra + OPM powers.
      4. Writes to HDF5 immediately and flushes — safe to interrupt.

    Resumable: re-running skips any already-computed sweep point
    (detected via flags/computed in the HDF5 file).

    Returns
    -------
    dict with keys:
        neff_sweep      np.ndarray (n_pts,)
        ng_sweep        np.ndarray (n_pts,)
        wavelengths_m   np.ndarray (n_wl,)
        T_port1_dB      np.ndarray (n_pts, n_wl)
        T_port2_dB      np.ndarray (n_pts, n_wl)
        opm_power_dBm   np.ndarray (n_pts, n_opm)
        opm_spectrum_W  np.ndarray (n_pts, n_opm, n_wl)
        computed        np.ndarray (n_pts,) bool
    """
    n_pts  = SWEEP_N_POINTS
    n_opm  = N_RINGS - 1

    # ── Allocate in-memory arrays (shape unknown until first run) ─────────────
    wavelengths_m  : Optional[np.ndarray] = None
    T_port1_dB     : Optional[np.ndarray] = None
    T_port2_dB     : Optional[np.ndarray] = None
    opm_power_dBm  : Optional[np.ndarray] = None
    opm_spectrum_W : Optional[np.ndarray] = None
    computed                               = np.zeros(n_pts, dtype=bool)
    hdf5_ready                             = False

    # ── Cache check ───────────────────────────────────────────────────────────
    if HDF5_PATH.exists():
        log.info(f"Cache found → {HDF5_PATH}")
        with h5py.File(HDF5_PATH, "r") as f:
            wavelengths_m   = f["metadata/wavelengths_m"][:]
            T_port1_dB      = f["results/T_port1_dB"][:]
            T_port2_dB      = f["results/T_port2_dB"][:]
            opm_power_dBm   = f["results/opm_power_dBm"][:]
            opm_spectrum_W  = f["results/opm_spectrum_W"][:]
            computed[:]     = f["flags/computed"][:]
        hdf5_ready = True
        n_cached   = int(computed.sum())
        remaining  = n_pts - n_cached
        log.info(f"Cached: {n_cached}/{n_pts}  |  Remaining: {remaining}")
        if remaining == 0:
            log.info("All sweep points cached — returning stored results.")
            return _pack_results(
                wavelengths_m, T_port1_dB, T_port2_dB,
                opm_power_dBm, opm_spectrum_W, computed
            )
    else:
        log.info("No cache found — starting fresh.")

    # ── Open INTERCONNECT session ─────────────────────────────────────────────
    log.info("Launching Lumerical INTERCONNECT …")
    ic = lumapi.INTERCONNECT(hide=hide_gui)

    runs_done  = 0
    runs_total = int((~computed).sum())
    t_start    = time.time()

    try:
        # Build circuit once; geometry never changes between sweep points
        _build_circuit(ic)
        log.info(f"Circuit ready — {runs_total} sweep point(s) remaining …")

        for s_idx in range(n_pts):

            if computed[s_idx]:
                continue

            neff_val = float(SWEEP_NEFF[s_idx])
            ng_val   = float(SWEEP_NG[s_idx])

            # ── Update Ring 1 parameters only ─────────────────────────────────
            ic.switchtolayout()
            _apply_ring_params(ic, ring_idx=0,
                               neff_override=neff_val,
                               ng_override=ng_val)

            # ── Run ───────────────────────────────────────────────────────────
            try:
                ic.run()
            except Exception as exc:
                log.warning(
                    f"  FAILED │ pt {s_idx} │ neff={neff_val:.5f} │ {exc}"
                )
                computed[s_idx] = True
                if hdf5_ready:
                    with h5py.File(HDF5_PATH, "r+") as hf:
                        hf["flags/computed"][s_idx] = True
                        hf.flush()
                continue

            # ── Extract ───────────────────────────────────────────────────────
            try:
                wl_m, t1, t2, opm_p, opm_s = _extract_results(ic)
            except Exception as exc:
                log.warning(
                    f"  EXTRACT FAILED │ pt {s_idx} │ {exc}"
                )
                computed[s_idx] = True
                continue

            # ── Initialise arrays and HDF5 on first successful run ────────────
            if wavelengths_m is None:
                n_wl           = len(wl_m)
                wavelengths_m  = wl_m
                T_port1_dB     = np.full((n_pts, n_wl),        np.nan)
                T_port2_dB     = np.full((n_pts, n_wl),        np.nan)
                opm_power_dBm  = np.full((n_pts, n_opm),       np.nan)
                opm_spectrum_W = np.full((n_pts, n_opm, n_wl), np.nan)
                # Load any already-cached rows if coming from partial HDF5
                if HDF5_PATH.exists() and not hdf5_ready:
                    with h5py.File(HDF5_PATH, "r") as f_old:
                        T_port1_dB     = f_old["results/T_port1_dB"][:]
                        T_port2_dB     = f_old["results/T_port2_dB"][:]
                        opm_power_dBm  = f_old["results/opm_power_dBm"][:]
                        opm_spectrum_W = f_old["results/opm_spectrum_W"][:]
                if not hdf5_ready:
                    _init_hdf5(wl_m)
                    hdf5_ready = True

            # ── Store in memory ───────────────────────────────────────────────
            T_port1_dB    [s_idx, :]    = t1
            T_port2_dB    [s_idx, :]    = t2
            opm_power_dBm [s_idx, :]    = opm_p
            opm_spectrum_W[s_idx, :, :] = opm_s
            computed      [s_idx]       = True

            # ── Write to HDF5 immediately (flush after every point) ───────────
            with h5py.File(HDF5_PATH, "r+") as hf:
                hf["results/T_port1_dB"]    [s_idx, :]    = t1
                hf["results/T_port2_dB"]    [s_idx, :]    = t2
                hf["results/opm_power_dBm"] [s_idx, :]    = opm_p
                hf["results/opm_spectrum_W"][s_idx, :, :] = opm_s
                hf["flags/computed"]        [s_idx]       = True
                hf.flush()

            runs_done += 1

            # ── Progress log ──────────────────────────────────────────────────
            if runs_done % 5 == 0 or runs_done == runs_total:
                elapsed = time.time() - t_start
                rate    = runs_done / elapsed if elapsed > 0 else 1e-9
                eta     = (runs_total - runs_done) / rate
                log.info(
                    f"  [{runs_done:3d}/{runs_total}]  "
                    f"neff = {neff_val:.5f}  ng = {ng_val:.5f}  │  "
                    f"{rate:.2f} sim/s  │  ETA {eta:5.0f} s"
                )

        # ── Finalise HDF5 metadata ────────────────────────────────────────────
        if hdf5_ready:
            with h5py.File(HDF5_PATH, "r+") as hf:
                hf["metadata"].attrs["timestamp_end"]  = datetime.now().isoformat()
                hf["metadata"].attrs["runs_completed"] = int(computed.sum())

    finally:
        ic.close()
        log.info("INTERCONNECT session closed.")

    elapsed_total = time.time() - t_start
    log.info(
        f"Sweep done │ {runs_done} new runs │ "
        f"total elapsed = {elapsed_total:.1f} s │ "
        f"avg = {elapsed_total / max(runs_done, 1):.2f} s/sim"
    )

    return _pack_results(
        wavelengths_m, T_port1_dB, T_port2_dB,
        opm_power_dBm, opm_spectrum_W, computed
    )


def _pack_results(wl, t1, t2, opm_p, opm_s, comp) -> Dict[str, Any]:
    """Bundle results into the standard return dict."""
    return dict(
        neff_sweep     = SWEEP_NEFF,
        ng_sweep       = SWEEP_NG,
        wavelengths_m  = wl,
        T_port1_dB     = t1,
        T_port2_dB     = t2,
        opm_power_dBm  = opm_p,
        opm_spectrum_W = opm_s,
        computed       = comp,
    )


# ─────────────────────────────────────────────────────────────────────────────
# EXECUTE CELL 3
# ─────────────────────────────────────────────────────────────────────────────
sweep_results = run_interconnect_sweep(hide_gui=False)

# ── Unpack into named variables for Cell 4 and beyond ────────────────────────
neff_sweep     = sweep_results["neff_sweep"]      # (n_pts,)
ng_sweep       = sweep_results["ng_sweep"]        # (n_pts,)
wavelengths_m  = sweep_results["wavelengths_m"]   # (n_wl,)
T_port1_dB     = sweep_results["T_port1_dB"]      # (n_pts, n_wl)
T_port2_dB     = sweep_results["T_port2_dB"]      # (n_pts, n_wl)
opm_power_dBm  = sweep_results["opm_power_dBm"]   # (n_pts, n_opm)
opm_spectrum_W = sweep_results["opm_spectrum_W"]  # (n_pts, n_opm, n_wl)
computed       = sweep_results["computed"]        # (n_pts,)  bool
wavelengths_nm = wavelengths_m * 1e9              # convenience alias

print(f"\n  Sweep complete — {computed.sum()} / {len(computed)} pts computed")
print(f"  T_port1_dB shape   : {T_port1_dB.shape}")
print(f"  opm_power_dBm shape: {opm_power_dBm.shape}")
print(f"  HDF5               : {HDF5_PATH}")

# ═════════════════════════════════════════════════════════════════════════════
#  END CELL 3
# ═════════════════════════════════════════════════════════════════════════════


# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 4 — Post-processing · Visualisation · Cache-aware re-plotting        ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

# ─────────────────────────────────────────────────────────────────────────────
# Matplotlib style
# ─────────────────────────────────────────────────────────────────────────────
plt.rcParams.update({
    "font.family"    : "serif",
    "font.serif"     : ["DejaVu Serif", "Georgia", "Times New Roman"],
    "font.size"      : 11,
    "axes.labelsize" : 12,
    "axes.titlesize" : 13,
    "legend.fontsize": 9,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "axes.linewidth" : 0.8,
    "axes.grid"      : True,
    "grid.alpha"     : 0.3,
    "grid.linewidth" : 0.5,
    "lines.linewidth": 1.4,
    "figure.dpi"     : 120,
    "savefig.dpi"    : 300,
    "savefig.bbox"   : "tight",
})


# ─────────────────────────────────────────────────────────────────────────────
# Data accessor  (cache-aware: loads from HDF5 if kernel was restarted)
# ─────────────────────────────────────────────────────────────────────────────
def load_results(path: Path = HDF5_PATH) -> Dict[str, Any]:
    """Load sweep results from HDF5. Used when Cell 3 was not run this session."""
    if not path.exists():
        raise FileNotFoundError(f"HDF5 file not found: {path}")
    with h5py.File(path, "r") as f:
        return dict(
            neff_sweep     = f["metadata/neff_sweep"][:],
            ng_sweep       = f["metadata/ng_sweep"][:],
            wavelengths_m  = f["metadata/wavelengths_m"][:],
            T_port1_dB     = f["results/T_port1_dB"][:],
            T_port2_dB     = f["results/T_port2_dB"][:],
            opm_power_dBm  = f["results/opm_power_dBm"][:],
            opm_spectrum_W = f["results/opm_spectrum_W"][:],
            computed       = f["flags/computed"][:],
        )


def get_results(path: Path = HDF5_PATH) -> Dict[str, Any]:
    """
    Return results dict from in-memory sweep_results if available,
    otherwise load from HDF5.  Makes Cell 4 safe to re-run after kernel restart.
    """
    try:
        r = sweep_results
        if r.get("wavelengths_m") is not None:
            return r
    except NameError:
        pass
    log.info("sweep_results not in memory — loading from HDF5.")
    return load_results(path)


# ─────────────────────────────────────────────────────────────────────────────
# Shared utility: _valid_mask
# ─────────────────────────────────────────────────────────────────────────────
def _valid_mask(results: Dict) -> np.ndarray:
    """Boolean mask of computed (non-NaN) sweep points."""
    return results["computed"].astype(bool)


# ─────────────────────────────────────────────────────────────────────────────
# Plot 1 — ONA Transmission spectra  (selected neff values, colour-mapped)
# ─────────────────────────────────────────────────────────────────────────────
def plot_transmittance_sweep(
    results     : Optional[Dict] = None,
    port        : int            = 1,
    n_curves    : int            = 8,
    figsize     : tuple          = (10, 5),
    cmap_name   : str            = "plasma",
    save        : bool           = True,
) -> plt.Figure:
    """
    Overlay ONA transmission spectra for n_curves evenly-spaced sweep points.
    Lines are colour-mapped to Ring 1 neff value.

    port=1 → RING_1 through (ONA input 1)
    port=2 → RING_14 through (ONA input 2)
    """
    if results is None:
        results = get_results()

    neff_arr  = results["neff_sweep"]
    wl_nm     = results["wavelengths_m"] * 1e9
    T_data    = results["T_port1_dB"] if port == 1 else results["T_port2_dB"]
    mask      = _valid_mask(results)

    valid_idx = np.where(mask)[0]
    n_sel     = min(n_curves, len(valid_idx))
    sel_idx   = valid_idx[
        np.round(np.linspace(0, len(valid_idx) - 1, n_sel)).astype(int)
    ]

    cmap = plt.get_cmap(cmap_name)
    norm = Normalize(vmin=neff_arr[sel_idx].min(), vmax=neff_arr[sel_idx].max())
    sm   = ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])

    fig, ax = plt.subplots(figsize=figsize)
    for idx in sel_idx:
        ax.plot(wl_nm, T_data[idx], color=cmap(norm(neff_arr[idx])),
                lw=1.2, alpha=0.88)

    cbar = fig.colorbar(sm, ax=ax, pad=0.02)
    cbar.set_label("Ring 1 — $n_{eff}$ (TE)", fontsize=11)

    port_label = "Ring 1 through  [ONA port 1]" if port == 1 \
                 else "Ring 14 through  [ONA port 2]"
    ax.set_xlabel("Wavelength  (nm)")
    ax.set_ylabel("Transmission  (dB)")
    ax.set_title(
        f"ONA Transmission — {port_label}\n"
        f"({n_sel} of {int(mask.sum())} sweep pts shown,  "
        f"colour = $n_{{eff,1}}$)"
    )
    ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
    ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())
    fig.tight_layout()

    if save:
        stem = f"transmittance_port{port}_{n_sel}curves"
        fig.savefig(FIGURES_DIR / f"{stem}.png")
        fig.savefig(FIGURES_DIR / f"{stem}.pdf")
        log.info(f"Saved → {FIGURES_DIR / stem}.png/pdf")

    return fig


# ─────────────────────────────────────────────────────────────────────────────
# Plot 2 — OPM mean power vs neff  (all 13 OPMs, one line each)
# ─────────────────────────────────────────────────────────────────────────────
def plot_opm_power_vs_neff(
    results    : Optional[Dict] = None,
    figsize    : tuple          = (11, 5),
    cmap_name  : str            = "tab20",
    save       : bool           = True,
) -> plt.Figure:
    """
    Plot mean drop power [dBm] at each OPM as a function of Ring 1 neff.
    All 13 OPMs overlaid on a single axes.
    """
    if results is None:
        results = get_results()

    neff_arr  = results["neff_sweep"]
    opm_power = results["opm_power_dBm"]   # (n_pts, n_opm)
    mask      = _valid_mask(results)

    neff_v   = neff_arr[mask]
    opm_v    = opm_power[mask, :]
    n_opm    = opm_v.shape[1]

    cmap     = plt.get_cmap(cmap_name)
    colors   = [cmap(i / n_opm) for i in range(n_opm)]

    fig, ax = plt.subplots(figsize=figsize)
    for k in range(n_opm):
        ring_id = k + 2
        ax.plot(
            neff_v, opm_v[:, k],
            color=colors[k], lw=1.3,
            marker="o", ms=3.5, markevery=max(1, len(neff_v) // 12),
            label=f"OPM {k+1}  (Ring {ring_id} drop)",
        )

    ax.set_xlabel("Ring 1 — $n_{eff}$ (TE)")
    ax.set_ylabel("Mean power  (dBm)")
    ax.set_title(
        "OPM Mean Drop Power  vs  Ring 1 $n_{eff}$\n"
        "(Each line = one ring's drop port, integrated over ONA bandwidth)"
    )
    ax.legend(loc="upper right", ncol=2, framealpha=0.85, fontsize=8)
    ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
    ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())
    fig.tight_layout()

    if save:
        fig.savefig(FIGURES_DIR / "opm_power_vs_neff.png")
        fig.savefig(FIGURES_DIR / "opm_power_vs_neff.pdf")
        log.info(f"Saved → {FIGURES_DIR / 'opm_power_vs_neff'}.png/pdf")

    return fig


# ─────────────────────────────────────────────────────────────────────────────
# Plot 3 — OPM spectral power heatmap  (neff × wavelength, per OPM)
# ─────────────────────────────────────────────────────────────────────────────
def plot_opm_spectrum_heatmap(
    results         : Optional[Dict]       = None,
    opm_ids         : Optional[List[int]]  = None,   # 1-indexed; None = all 13
    figsize_per_row : tuple                = (10, 2.6),
    cmap_name       : str                  = "inferno",
    vmin_dBm        : Optional[float]      = None,
    vmax_dBm        : Optional[float]      = None,
    save            : bool                 = True,
) -> plt.Figure:
    """
    2D heatmap: X = wavelength [nm], Y = Ring 1 neff, colour = power [dBm].
    One subplot per selected OPM.  Shows resonance band shifting with neff.
    """
    if results is None:
        results = get_results()

    neff_arr = results["neff_sweep"]
    wl_nm    = results["wavelengths_m"] * 1e9
    opm_spec = results["opm_spectrum_W"]   # (n_pts, n_opm, n_wl)
    mask     = _valid_mask(results)

    neff_v   = neff_arr[mask]
    spec_v   = opm_spec[mask, :, :]
    n_opm    = spec_v.shape[1]

    if opm_ids is None:
        opm_ids = list(range(1, n_opm + 1))

    # Convert W → dBm
    spec_dBm = 10.0 * np.log10(np.where(spec_v > 0, spec_v, 1e-30) * 1e3)
    _vmin = vmin_dBm if vmin_dBm is not None else np.nanpercentile(spec_dBm, 2)
    _vmax = vmax_dBm if vmax_dBm is not None else np.nanpercentile(spec_dBm, 98)

    n_rows = len(opm_ids)
    fig, axes = plt.subplots(
        n_rows, 1,
        figsize=(figsize_per_row[0], figsize_per_row[1] * n_rows),
        sharex=True, sharey=True,
    )
    if n_rows == 1:
        axes = [axes]

    for ax_i, opm_id in enumerate(opm_ids):
        k   = opm_id - 1
        img = axes[ax_i].pcolormesh(
            wl_nm, neff_v, spec_dBm[:, k, :],
            cmap=cmap_name, vmin=_vmin, vmax=_vmax, shading="auto",
        )
        ring_id = k + 2
        axes[ax_i].set_ylabel(f"OPM {opm_id}\nRing {ring_id}\n$n_{{eff,1}}$",
                              fontsize=9)
        cbar = fig.colorbar(img, ax=axes[ax_i], pad=0.01)
        cbar.set_label("dBm", fontsize=8)
        cbar.ax.tick_params(labelsize=8)

    axes[-1].set_xlabel("Wavelength  (nm)")
    fig.suptitle(
        "OPM Spectral Power  vs  Ring 1 $n_{eff}$\n"
        "(Bright band = resonance; tracks shift as neff changes)",
        fontsize=12, y=1.01,
    )
    fig.tight_layout()

    if save:
        tag   = "_".join(str(o) for o in opm_ids)
        fname = f"opm_spectrum_heatmap_{tag}"
        fig.savefig(FIGURES_DIR / f"{fname}.png")
        fig.savefig(FIGURES_DIR / f"{fname}.pdf")
        log.info(f"Saved → {FIGURES_DIR / fname}.png/pdf")

    return fig


# ─────────────────────────────────────────────────────────────────────────────
# Plot 4 — Resonance wavelength tracking  vs  neff  (ONA transmission dip)
# ─────────────────────────────────────────────────────────────────────────────
def plot_resonance_tracking(
    results : Optional[Dict] = None,
    port    : int            = 1,
    figsize : tuple          = (7, 4.5),
    save    : bool           = True,
) -> plt.Figure:
    """
    Locate the deepest transmission dip at each sweep point and plot
    resonance wavelength vs Ring 1 neff.  Linear fit gives ∂λ/∂neff.
    """
    if results is None:
        results = get_results()

    neff_arr = results["neff_sweep"]
    wl_nm    = results["wavelengths_m"] * 1e9
    T_data   = results["T_port1_dB"] if port == 1 else results["T_port2_dB"]
    mask     = _valid_mask(results)

    neff_v  = neff_arr[mask]
    T_v     = T_data[mask, :]

    dip_idx  = np.argmin(T_v, axis=1)
    lam_dip  = wl_nm[dip_idx]

    coeffs  = np.polyfit(neff_v, lam_dip, 1)
    poly    = np.poly1d(coeffs)
    sens    = coeffs[0]   # nm / RIU

    fig, ax = plt.subplots(figsize=figsize)
    ax.scatter(neff_v, lam_dip, s=20, zorder=5, color="#2166ac",
               label="Resonance dip")
    ax.plot(neff_v, poly(neff_v), "r--", lw=1.5,
            label=f"Linear fit   ∂λ/∂n = {sens:.3f} nm / RIU")

    ax.set_xlabel("Ring 1 — $n_{eff}$ (TE)")
    ax.set_ylabel("Resonance wavelength  (nm)")
    ax.set_title(
        f"Resonance Wavelength Tracking — ONA Port {port}\n"
        f"Sensitivity: {sens:.3f} nm / RIU"
    )
    ax.legend(framealpha=0.9)
    ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
    ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())
    fig.tight_layout()

    if save:
        fname = f"resonance_tracking_port{port}"
        fig.savefig(FIGURES_DIR / f"{fname}.png")
        fig.savefig(FIGURES_DIR / f"{fname}.pdf")
        log.info(f"Saved → {FIGURES_DIR / fname}.png/pdf")

    log.info(f"Resonance sensitivity (port {port}): {sens:.4f} nm / RIU")
    return fig


# ─────────────────────────────────────────────────────────────────────────────
# Plot registry  — clean hook for future cells
# ─────────────────────────────────────────────────────────────────────────────
PLOT_REGISTRY: Dict[str, Any] = {
    "transmittance_port1"   : lambda **kw: plot_transmittance_sweep(port=1, **kw),
    "transmittance_port2"   : lambda **kw: plot_transmittance_sweep(port=2, **kw),
    "opm_power_vs_neff"     : plot_opm_power_vs_neff,
    "opm_spectrum_heatmap"  : plot_opm_spectrum_heatmap,
    "resonance_tracking_p1" : lambda **kw: plot_resonance_tracking(port=1, **kw),
    "resonance_tracking_p2" : lambda **kw: plot_resonance_tracking(port=2, **kw),
}

# ─────────────────────────────────────────────────────────────────────────────
# Dependency registry  — exported handles for future cells
# ─────────────────────────────────────────────────────────────────────────────
DEPENDENCY_REGISTRY: Dict[str, Any] = {
    # Element naming
    "ring_name"                    : ring_name,
    "opm_name"                     : opm_name,
    "ONA_NAME"                     : ONA_NAME,
    # INTERCONNECT helpers
    "_apply_ring_params"           : _apply_ring_params,
    "_build_circuit"               : _build_circuit,
    "_extract_results"             : _extract_results,
    "_init_hdf5"                   : _init_hdf5,
    # Sweep engine
    "run_interconnect_sweep"       : run_interconnect_sweep,
    # Data I/O
    "load_results"                 : load_results,
    "get_results"                  : get_results,
    # Plots
    "PLOT_REGISTRY"                : PLOT_REGISTRY,
    "plot_transmittance_sweep"     : plot_transmittance_sweep,
    "plot_opm_power_vs_neff"       : plot_opm_power_vs_neff,
    "plot_opm_spectrum_heatmap"    : plot_opm_spectrum_heatmap,
    "plot_resonance_tracking"      : plot_resonance_tracking,
    # Paths
    "DATA_DIR"                     : DATA_DIR,
    "FIGURES_DIR"                  : FIGURES_DIR,
    "HDF5_PATH"                    : HDF5_PATH,
    "VERSION_NAME"                 : VERSION_NAME,
    "N_RINGS"                      : N_RINGS,
}

# ─────────────────────────────────────────────────────────────────────────────
# EXECUTE CELL 4
# ─────────────────────────────────────────────────────────────────────────────
_res = get_results()

fig1 = plot_transmittance_sweep(_res, port=1, n_curves=8)
fig2 = plot_transmittance_sweep(_res, port=2, n_curves=8)
fig3 = plot_opm_power_vs_neff(_res)
fig4 = plot_opm_spectrum_heatmap(_res, opm_ids=[1, 4, 7, 10, 13])
fig5 = plot_resonance_tracking(_res, port=1)

plt.show()

print(f"\n  All figures saved to: {FIGURES_DIR}")
print(f"  HDF5 data at        : {HDF5_PATH}")
print(f"\n  Dependency registry keys:")
for k in DEPENDENCY_REGISTRY:
    print(f"    {k}")

# ═════════════════════════════════════════════════════════════════════════════
#  END CELL 4
# ═════════════════════════════════════════════════════════════════════════════
#
#  ── PATTERNS FOR FUTURE CELLS ──────────────────────────────────────────────
#
#  1. Change a ring's coupling coefficient and re-run:
#       RING_KAPPA_INPUT_SQ[2] = 0.08   # Ring 3
#       sweep_results = run_interconnect_sweep()
#
#  2. Add a second sweep over ring radius (extend the loop in Cell 3):
#       for r_val in np.linspace(4e-6, 7e-6, 5):
#           RING_RADIUS_M[0] = r_val
#           results_r = run_interconnect_sweep()
#
#  3. Re-plot from HDF5 after kernel restart (no simulation needed):
#       r = load_results(HDF5_PATH)
#       plot_opm_power_vs_neff(r)
#
#  4. Run all plots from the registry:
#       for name, fn in PLOT_REGISTRY.items():
#           fn(results=_res, save=True)
#
#  5. Access per-ring parameters by index in downstream cells:
#       print(RING_NEFF_TE[0])    # Ring 1
#       print(RING_KAPPA_DROP_SQ) # all 14 values
# ═════════════════════════════════════════════════════════════════════════════

20:05:41 │ INFO │ No cache found — starting fresh.
20:05:41 │ INFO │ Launching Lumerical INTERCONNECT …


lumapi imported successfully from:
  C:\Program Files\Lumerical\v202\api\python\lumapi.py

  Data directory : C:\Users\jero0\OneDrive\Escritorio\Github\GDS_py_TDY_venv\Fabrication_designs\Lumerical_scripts\data_ICNT_cascade_ring_sweep
  HDF5 output    : C:\Users\jero0\OneDrive\Escritorio\Github\GDS_py_TDY_venv\Fabrication_designs\Lumerical_scripts\data_ICNT_cascade_ring_sweep\ICNT_14Ring_Cascade_neff_sweep_V1.h5
  Figures dir    : C:\Users\jero0\OneDrive\Escritorio\Github\GDS_py_TDY_venv\Fabrication_designs\Lumerical_scripts\data_ICNT_cascade_ring_sweep\figures
  INTERCONNECT 14-Ring Cascade — Parameter Summary
   Ring    R [µm]    λ_res [nm]   neff_TE    ng_TE    κ²_in    κ²_dr  Loss [dB/m]
  ─────────────────────────────────────────────────────────────────
      1    19.002      1550.000   1.60980  2.02054   0.1458   0.1434        101.0 ◄ swept
      2    19.182      1550.000   1.63330  1.99110   0.1451   0.1427        101.0
      3    19.193      1550.769   1.63312  1.99096   0.1450

20:05:50 │ INFO │ INTERCONNECT session closed.


AttributeError: 'INTERCONNECT' object has no attribute 'addona'